In [2]:
pip install PyPDF2

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import PyPDF2
import re

def find_start_page(pdf_path, fallback_page):
    """
    Busca la primera página donde empieza realmente la ley (Artículo 1).
    Si no la encuentra por texto, usa una página por defecto (fallback).
    """
    try:
        with open(pdf_path, 'rb') as f:
            reader = PyPDF2.PdfReader(f)
            for i in range(len(reader.pages)):
                text = reader.pages[i].extract_text()
                if text:
                    # Buscamos el patrón típico de inicio: "Artículo 1" seguido de "Objeto"
                    if re.search(r'Artículo\s+1\b[\s\S]{0,100}Objeto', text, re.IGNORECASE) or \
                       re.search(r'CAP[ÍI]TULO\s+I[\s\S]{0,100}Artículo\s+1\b', text, re.IGNORECASE):
                        print(f"[{pdf_path}] 'Artículo 1' encontrado en la página física {i + 1}.")
                        return i
    except Exception as e:
        print(f"Error leyendo {pdf_path}: {e}")
        
    print(f"[{pdf_path}] No se encontró mediante IA, usando página por defecto: {fallback_page + 1}")
    return fallback_page

def recortar_pdf(input_path, output_path, start_page):
    """
    Crea un nuevo PDF descartando todas las páginas anteriores a 'start_page'.
    """
    with open(input_path, 'rb') as f_in:
        reader = PyPDF2.PdfReader(f_in)
        writer = PyPDF2.PdfWriter()
        
        # Añadimos desde la página de inicio exacta hasta el final del documento
        for i in range(start_page, len(reader.pages)):
            writer.add_page(reader.pages[i])
            
        with open(output_path, 'wb') as f_out:
            writer.write(f_out)
    print(f"✅ Éxito: '{output_path}' generado (Amputadas las primeras {start_page} páginas de preámbulo).")

# ==========================================
# EJECUCIÓN DEL PIPELINE DE PRE-PROCESAMIENTO
# ==========================================

# 1. Recortar el RGPD (GDPR) - Por seguridad, si falla el escáner, corta en la pág 32
gdpr_start = find_start_page('../pdfs/base/gdpr.pdf', fallback_page=31)
recortar_pdf('../pdfs/base/gdpr.pdf', '../pdfs/recortados/gdpr_recortado.pdf', gdpr_start)

# 2. Recortar la AI Act - Por seguridad, corta en la pág 44
aiact_start = find_start_page('../pdfs/base/ai_act.pdf', fallback_page=43)
recortar_pdf('../pdfs/base/ai_act.pdf', '../pdfs/recortados/aiact_recortado.pdf', aiact_start)

aiact_start = find_start_page('../pdfs/base/nis2.pdf', fallback_page=28)
recortar_pdf('../pdfs/base/nis2.pdf', '../pdfs/recortados/nis2_recortado.pdf', aiact_start)

[../pdfs/base/gdpr.pdf] 'Artículo 1' encontrado en la página física 32.
✅ Éxito: '../pdfs/recortados/gdpr_recortado.pdf' generado (Amputadas las primeras 31 páginas de preámbulo).
[../pdfs/base/ai_act.pdf] 'Artículo 1' encontrado en la página física 44.
✅ Éxito: '../pdfs/recortados/aiact_recortado.pdf' generado (Amputadas las primeras 43 páginas de preámbulo).
[../pdfs/base/nis2.pdf] 'Artículo 1' encontrado en la página física 29.
✅ Éxito: '../pdfs/recortados/nis2_recortado.pdf' generado (Amputadas las primeras 28 páginas de preámbulo).
